In [1]:
import shutil
import random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import yaml

INPUT_ROOT = Path('/kaggle/input')
WORK_DIR = Path('/kaggle/working/dataset')

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)

for split in ['train', 'val']:
    (WORK_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (WORK_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

# 1. Xác định đường dẫn chính xác của 2 bộ dữ liệu
sample_wider = next(INPUT_ROOT.rglob('wider_*.jpg'), None)
assert sample_wider is not None, "Chưa gắn bộ WIDER FACE vào Input!"
wider_dir = sample_wider.parent.parent

esp_train_imgs = [p for p in INPUT_ROOT.rglob('train/images/*.*') if not str(p).startswith(str(wider_dir))]
assert len(esp_train_imgs) > 0, "Không tìm thấy thư mục train/images trong bộ ESP32!"
esp_dir = esp_train_imgs[0].parent.parent.parent
esp_val_imgs = list(esp_dir.glob('valid/images/*.*')) or list(esp_dir.glob('val/images/*.*'))

print(f"[1/4] Đã tìm thấy {len(esp_train_imgs)} ảnh ESP32 train và {len(esp_val_imgs)} ảnh ESP32 val.")

# 2. Tạo danh sách copy và nhân bản 5 lần tập ESP32
copy_tasks = []
for img_path in esp_train_imgs:
    lbl_path = Path(str(img_path).replace('/images/', '/labels/')).with_suffix('.txt')
    if lbl_path.exists():
        for i in range(5):
            new_name = f"esp_c{i}_{img_path.name}"
            copy_tasks.append((img_path, WORK_DIR / 'images/train' / new_name))
            copy_tasks.append((lbl_path, WORK_DIR / 'labels/train' / Path(new_name).with_suffix('.txt')))

for img_path in esp_val_imgs:
    lbl_path = Path(str(img_path).replace('/images/', '/labels/')).with_suffix('.txt')
    if lbl_path.exists():
        copy_tasks.append((img_path, WORK_DIR / 'images/val' / f"esp_{img_path.name}"))
        copy_tasks.append((lbl_path, WORK_DIR / 'labels/val' / f"esp_{lbl_path.stem}.txt"))

# 3. Lọc đa luồng WIDER FACE: Chỉ giữ ảnh 1-3 người và mặt to rõ
wider_all_imgs = list((wider_dir / 'images').glob('*.jpg'))
print(f"[2/4] Đang lọc song song {len(wider_all_imgs)} ảnh WIDER FACE...")

def check_wider_item(img_path):
    lbl_path = (wider_dir / 'labels' / img_path.name).with_suffix('.txt')
    if not lbl_path.exists():
        return None
    lines = [l.strip() for l in lbl_path.read_text().splitlines() if l.strip()]
    if not (1 <= len(lines) <= 3):
        return None
    for line in lines:
        parts = line.split()
        if len(parts) >= 5 and (float(parts[3]) < 0.08 or float(parts[4]) < 0.10):
            return None
    return (img_path, lbl_path)

with ThreadPoolExecutor(max_workers=16) as ex:
    filtered_wider = [r for r in ex.map(check_wider_item, wider_all_imgs) if r is not None]

print(f"[3/4] Lọc xong! Giữ lại {len(filtered_wider)} ảnh phù hợp. Đang copy sang thư mục train...")
random.seed(42)
random.shuffle(filtered_wider)

for img_path, lbl_path in filtered_wider[:2000]:
    copy_tasks.append((img_path, WORK_DIR / 'images/train' / f"wider_{img_path.name}"))
    copy_tasks.append((lbl_path, WORK_DIR / 'labels/train' / f"wider_{lbl_path.stem}.txt"))

for img_path, lbl_path in filtered_wider[2000:2250]:
    copy_tasks.append((img_path, WORK_DIR / 'images/val' / f"wider_{img_path.name}"))
    copy_tasks.append((lbl_path, WORK_DIR / 'labels/val' / f"wider_{lbl_path.stem}.txt"))

with ThreadPoolExecutor(max_workers=16) as ex:
    ex.map(lambda pair: shutil.copyfile(pair[0], pair[1]), copy_tasks)

print(f"[4/4] HOÀN TẤT! Tổng ảnh Train: {len(list((WORK_DIR / 'images/train').glob('*')))} | Val: {len(list((WORK_DIR / 'images/val').glob('*')))}")

# 4. Xuất file cấu hình data.yaml
data_yaml = {
    'path': str(WORK_DIR),
    'train': 'images/train',
    'val': 'images/val',
    'nc': 1,
    'names': ['face']
}
with open('/kaggle/working/data.yaml', 'w') as f:
    yaml.dump(data_yaml, f)

[1/4] Đã tìm thấy 156 ảnh ESP32 train và 39 ảnh ESP32 val.
[2/4] Đang lọc song song 12880 ảnh WIDER FACE...
[3/4] Lọc xong! Giữ lại 4196 ảnh phù hợp. Đang copy sang thư mục train...
[4/4] HOÀN TẤT! Tổng ảnh Train: 2780 | Val: 289


In [2]:
!pip install -q ultralytics
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data='/kaggle/working/data.yaml',
    epochs=30,
    imgsz=320,
    batch=32,
    hsv_v=0.4,       # Tăng cường biến thiên độ sáng cho cam ESP32
    degrees=15.0,    # Xoay nhẹ ảnh để bắt tốt góc nghiêng đầu
    project='/kaggle/working/runs_det',
    name='esp32_face_det'
)

# Xuất model sang định dạng ONNX để chạy suy luận thực tế
best_model = YOLO('/kaggle/working/runs_det/esp32_face_det/weights/best.pt')
best_model.export(format='onnx', imgsz=320)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=15.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, free

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/30      1.48G      1.351     0.9098       1.29         81        320: 100% ━━━━━━━━━━━━ 87/87 8.3it/s 10.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 2.9it/s 1.7s
                   all        289        349      0.898      0.785      0.867      0.499

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/30      1.48G      1.322     0.8693      1.264         63        320: 100% ━━━━━━━━━━━━ 87/87 8.2it/s 10.6s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 3.0it/s 1.7s
                   all        289        349      0.926      0.742       0.84      0.509

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/30      1.48G      1.304     0.8492       1.26         60        320: 100% ━━━━━━━━━━━━ 87/87 8.3it/s 10.5s
                 Class     Images  Instances      Box(P        

'/kaggle/working/runs_det/esp32_face_det/weights/best.onnx'